# Exploratory Data Analysis — Cleaned Touchpoint Data

**Goal:** Characterize the cleaned dataset (post bot-removal) across funnel shape,
channel performance, campaign spread, journey complexity, and time patterns — both to
sanity-check the cleaning step and to surface patterns that inform the attribution
modeling in `Models.ipynb`.

**Input:** `touchpoints_clean_v3.csv` (372,570 rows / 98,048 users)
**Output:** `eda_channel_funnel.csv`

In [2]:
import pandas as pd

In [3]:
import numpy as np

In [6]:
df = pd.read_csv('touchpoints_clean_v3.csv', parse_dates=['timestamp'])
df['channel']    = df['channel'].str.strip().str.title()
df['event_type'] = df['event_type'].str.strip()

print(f"Clean dataset: {len(df):,} rows | {df['user_id'].nunique():,} users\n")

Clean dataset: 372,570 rows | 98,048 users



## EDA 1 — Overall funnel

Impression → Click → Add-to-Cart → Purchase counts and the conversion rate at each
step. The Add-to-Cart → Purchase rate above 100% is expected here — it isn't a data
error, it reflects that not every purchase requires (or is preceded by) a logged
Add-to-Cart event in this schema, so the two counts aren't a strict subset/superset
pair. The overall Impression → Purchase rate (~1.6%) is the headline funnel metric.

In [9]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 1 — FUNNEL OVERVIEW (the big picture)
# ══════════════════════════════════════════════════════════════════════════
print("="*70)
print("EDA 1 — OVERALL FUNNEL")
print("="*70)
funnel = df['event_type'].value_counts()
funnel_pct = df['event_type'].value_counts(normalize=True) * 100
funnel_df = pd.DataFrame({'count': funnel, 'pct': funnel_pct.round(2)})
print(funnel_df)

impressions = funnel.get('Impression', 0)
clicks      = funnel.get('Click', 0)
carts       = funnel.get('Add-to-Cart', 0)
purchases   = funnel.get('Purchase', 0)

print(f"\nFunnel conversion rates:")
print(f"  Impression -> Click:      {clicks/impressions*100:.2f}%")
print(f"  Click -> Add-to-Cart:     {carts/clicks*100:.2f}%")
print(f"  Add-to-Cart -> Purchase:  {purchases/carts*100:.2f}%")
print(f"  Overall Impression->Purchase: {purchases/impressions*100:.3f}%")

EDA 1 — OVERALL FUNNEL
              count    pct
event_type                
Impression   342724  91.99
Click         21221   5.70
Purchase       5498   1.48
Add-to-Cart    3127   0.84

Funnel conversion rates:
  Impression -> Click:      6.19%
  Click -> Add-to-Cart:     14.74%
  Add-to-Cart -> Purchase:  175.82%
  Overall Impression->Purchase: 1.604%


## EDA 2 — Channel-level breakdown

Click-through rate and click-to-purchase conversion rate per channel. CTR is fairly
flat across channels (5.97%–6.31%) — no channel is dramatically better at earning a
click. The real differentiation shows up in conversion rate *after* the click: Google
Search converts clicks to purchases far more often (62.1%) than Youtube or Influencer
Blog (~10–12%). This is consistent with Search capturing higher-intent, later-funnel
traffic rather than being intrinsically a "better" channel — a theme also borne out in
`Diagnose.ipynb`.

In [12]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 2 — CHANNEL-LEVEL BREAKDOWN
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("EDA 2 — EVENTS PER CHANNEL")
print("="*70)
channel_funnel = df.groupby(['channel', 'event_type']).size().unstack(fill_value=0)
# reorder columns logically
col_order = [c for c in ['Impression','Click','Add-to-Cart','Purchase'] if c in channel_funnel.columns]
channel_funnel = channel_funnel[col_order]
print(channel_funnel)

print("\nClick-through rate (CTR) per channel:")
ctr = (channel_funnel['Click'] / channel_funnel['Impression'] * 100).sort_values(ascending=False)
print(ctr.round(2))

print("\nConversion rate (Purchase / Click) per channel:")
conv = (channel_funnel['Purchase'] / channel_funnel['Click'] * 100).sort_values(ascending=False)
print(conv.round(2))


EDA 2 — EVENTS PER CHANNEL
event_type       Impression  Click  Add-to-Cart  Purchase
channel                                                  
Google Search         68807   4284          619      2661
Influencer Blog       68494   4253          639       501
Instagram             68619   4289          638       873
Marketplace           68441   4083          601      1024
Youtube               68363   4312          630       439

Click-through rate (CTR) per channel:
channel
Youtube            6.31
Instagram          6.25
Google Search      6.23
Influencer Blog    6.21
Marketplace        5.97
dtype: float64

Conversion rate (Purchase / Click) per channel:
channel
Google Search      62.11
Marketplace        25.08
Instagram          20.35
Influencer Blog    11.78
Youtube            10.18
dtype: float64


## EDA 3 — Campaign-level spread

Checks that campaign volume is reasonably even (no single campaign dominating the
dataset) and confirms the `brand_id` extraction from `campaign_id` works correctly
across all 10 brands, each with roughly comparable event volume (~36K–38K rows).

In [15]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 3 — CAMPAIGN-LEVEL SPREAD
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("EDA 3 — CAMPAIGN ACTIVITY")
print("="*70)
print(f"Unique campaigns: {df['campaign_id'].nunique()}")
camp_counts = df.groupby('campaign_id').size().sort_values(ascending=False)
print("\nTop 5 most active campaigns:")
print(camp_counts.head(5))
print("\nBottom 5 least active campaigns:")
print(camp_counts.tail(5))

# Extract brand from campaign_id if pattern like CMP_B07_GOO_945 (B07 = brand)
df['brand_id'] = df['campaign_id'].str.extract(r'_(B\d+)_')
print(f"\nUnique brands detected: {df['brand_id'].nunique()}")
print(df['brand_id'].value_counts().sort_index())



EDA 3 — CAMPAIGN ACTIVITY
Unique campaigns: 50

Top 5 most active campaigns:
campaign_id
CMP_B07_GOO_945    8602
CMP_B02_GOO_562    8451
CMP_B01_INS_285    7871
CMP_B07_MAR_355    7552
CMP_B03_YOU_269    7547
dtype: int64

Bottom 5 least active campaigns:
campaign_id
CMP_B02_YOU_280    7306
CMP_B08_INF_464    7273
CMP_B07_YOU_778    7252
CMP_B05_YOU_992    7226
CMP_B05_INS_247    7184
dtype: int64

Unique brands detected: 10
brand_id
B01    37501
B02    38000
B03    37307
B04    37173
B05    36486
B06    37143
B07    38113
B08    36804
B09    37114
B10    36929
Name: count, dtype: int64


## EDA 4 — Journey complexity (single- vs multi-channel)

Counts how many distinct channels each user touches. Only 20.8% of users are
single-channel; the remaining 79.2% touch 2 or more channels before their journey
ends. This is the key justification for using a multi-touch attribution model
(Markov / Shapley) rather than any single-touch rule — most user journeys genuinely
span multiple channels, so crediting only one channel would misrepresent how most
users actually convert.

In [17]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 4 — USER JOURNEY LENGTH (single vs multi-channel)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("EDA 4 — JOURNEY COMPLEXITY")
print("="*70)
user_channels = df.groupby('user_id')['channel'].apply(lambda x: x.nunique())
print("How many DIFFERENT channels does each user touch?")
print(user_channels.value_counts().sort_index())
print(f"\n% of users touching only 1 channel: {(user_channels==1).mean()*100:.2f}%")
print(f"% of users touching 2+ channels:     {(user_channels>=2).mean()*100:.2f}%")

user_events = df.groupby('user_id').size()
print(f"\nEvents per user (post-cleaning):")
print(user_events.describe())



EDA 4 — JOURNEY COMPLEXITY
How many DIFFERENT channels does each user touch?
channel
1    20364
2    26944
3    30878
4    17385
5     2477
Name: count, dtype: int64

% of users touching only 1 channel: 20.77%
% of users touching 2+ channels:     79.23%

Events per user (post-cleaning):
count    98048.000000
mean         3.799874
std          1.930132
min          1.000000
25%          2.000000
50%          4.000000
75%          5.000000
max         13.000000
dtype: float64


## EDA 5 — Time-based patterns

Checks for any strong time-of-day or day-of-week skew that might bias the dataset (or
suggest scheduled/bot-like batch activity, relevant context for the bot detection
work). Activity is reasonably spread across business hours and the week, with
Thursday showing notably higher volume — worth flagging as a pattern, though not
extreme enough on its own to indicate non-human traffic.

In [20]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 5 — TIME PATTERNS
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("EDA 5 — TIME-BASED PATTERNS")
print("="*70)
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()

print("Events by hour of day (top 5 busiest hours):")
print(df['hour'].value_counts().sort_values(ascending=False).head(5))

print("\nEvents by day of week:")
print(df['day_of_week'].value_counts())

print("\nPurchases by day of week (where do conversions cluster?):")
print(df[df['event_type']=='Purchase']['day_of_week'].value_counts())


EDA 5 — TIME-BASED PATTERNS
Events by hour of day (top 5 busiest hours):
hour
8     18611
9     18584
10    18408
11    18359
18    18216
Name: count, dtype: int64

Events by day of week:
day_of_week
Thursday     124543
Saturday      55818
Sunday        53574
Friday        44409
Monday        36517
Tuesday       32718
Wednesday     24991
Name: count, dtype: int64

Purchases by day of week (where do conversions cluster?):
day_of_week
Sunday       899
Saturday     895
Monday       826
Friday       824
Tuesday      758
Wednesday    649
Thursday     647
Name: count, dtype: int64


## EDA 6 — Post-cleaning data quality check

Final check that bot removal didn't introduce nulls or duplicate rows. Zero nulls and
zero exact duplicates confirm the cleaned dataset is structurally sound and ready to
feed into attribution modeling.

In [22]:
# ══════════════════════════════════════════════════════════════════════════
# EDA 6 — CHECK FOR NULLS / DUPLICATES POST-CLEANING
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("EDA 6 — DATA QUALITY POST-CLEANING")
print("="*70)
print("Nulls per column:")
print(df.isnull().sum())
print(f"\nExact duplicate rows: {df.duplicated().sum()}")

# ── SAVE SUMMARY ─────────────────────────────────────────────────────────────
channel_funnel.to_csv('eda_channel_funnel.csv')
print("\n✓ Saved: eda_channel_funnel.csv")


EDA 6 — DATA QUALITY POST-CLEANING
Nulls per column:
user_id        0
timestamp      0
campaign_id    0
channel        0
event_type     0
brand_id       0
hour           0
day_of_week    0
dtype: int64

Exact duplicate rows: 0

✓ Saved: eda_channel_funnel.csv
